# 03 — Evaluate rejection and predict
Choose the best validation checkpoint. Generate low-confidence predictions
for COCO AP, then select a threshold using validation only. The test cells
remain disabled until you freeze the model and operating point.


## Repository and Drive
In Colab choose a GPU runtime with Python 3.10–3.12. Upload a ZIP containing
this repository's code, notebooks, configs, requirements and tests, and extract
it to `/content/aquafina-yolo-detector` using the Files pane. Do not include
datasets, checkpoints, caches, or `.git`. This works without a commit or push.
After an approved future push, cloning your own repository is also an option.
The repository on your computer remains the authoritative code copy.


In [ ]:
from pathlib import Path
import os, sys, subprocess
from google.colab import drive
drive.mount('/content/drive')
REPO = Path('/content/aquafina-yolo-detector')
DRIVE = Path('/content/drive/MyDrive/aquafina-yolo')
assert (REPO / 'pyproject.toml').is_file(), 'Extract repository code first'
os.chdir(REPO)
sys.path.insert(0, str(REPO / 'src'))
os.environ['PYTHONPATH'] = str(REPO / 'src') + ':/content/YOLOX'
PREPARED = DRIVE / 'processed/temple'


## Install the pinned GPU environment
This downloads official YOLOX **source** and Python dependencies only.
If Colab reports that already-imported packages changed, restart the session,
rerun the Repository and Drive cell, then continue with the audit cell.
Do not import torch/numpy before this install cell in a fresh session.


In [ ]:
subprocess.run([sys.executable, '-m', 'aquafina_detector.bootstrap', '--repo', str(REPO)], check=True)


In [ ]:
sys.path.insert(0, '/content/YOLOX')
import torch
from aquafina_detector.bootstrap import audit_runtime
from aquafina_detector.common import write_json
audit = audit_runtime()
assert torch.__version__.split('+')[0] == '2.5.1'
assert torch.cuda.is_available(), 'Select a GPU runtime'
write_json(DRIVE / 'environment/audit.json', audit)
print(torch.cuda.get_device_name(0), torch.__version__)


In [ ]:
from aquafina_detector.predict import Predictor, predict_coco, predict_media
from aquafina_detector.evaluate import evaluate_files
from aquafina_detector.common import read_json
CHECKPOINT = DRIVE / 'runs/baseline-v1/best_ckpt.pth'
REPORTS = DRIVE / 'reports/baseline-v1'
REPORTS.mkdir(parents=True, exist_ok=True)
predictor = Predictor(CHECKPOINT, confidence=0.001, nms=0.65)
val_predictions = REPORTS / 'val_predictions.json'
predict_coco(predictor, PREPARED / 'annotations/val.json', PREPARED / 'val2017', val_predictions)
operating_point = REPORTS / 'operating_point.json'
result = evaluate_files(PREPARED / 'annotations/val.json', val_predictions, operating_point)
print(result)


If status is `no_qualifying_threshold`, do not claim successful brand rejection.
Review false positives on validation and collect additional independent training
examples. Never copy test images into training. Reports include counts, AP,
image-level false positive rates and separate subset results. Precision/FPR
constraints are provisional targets, not guarantees; inspect sample size.


In [ ]:
from PIL import Image, ImageDraw
from IPython.display import display
report = result.get('operating_report') or result.get('diagnostic_report')
if report:
    for error in report['errors'][:12]:
        print(error)
        display(Image.open(PREPARED / 'val2017' / error['file_name']))
else:
    print('No report available.')


## Held-out test — explicit final evaluation
Enable only after model/threshold freeze. The evaluator checks checkpoint,
preprocessing and NMS provenance. It never searches thresholds on test.


In [ ]:
RUN_FINAL_TEST = False
if RUN_FINAL_TEST:
    assert (PREPARED / 'annotations/test.json').is_file(), 'TempleRAIL has train/val only; supply an independent test set first'
    assert read_json(operating_point)['status'] == 'qualified'
    test_predictions = REPORTS / 'test_predictions.json'
    predict_coco(predictor, PREPARED / 'annotations/test.json', PREPARED / 'test2017', test_predictions)
    print(evaluate_files(PREPARED / 'annotations/test.json', test_predictions,
                         REPORTS / 'test_report.json', operating_point))


## Image or saved-video inference
Set INPUT to your Drive file and RUN_PREDICTION=True. Outputs include annotated
media and original-coordinate detections; video has frame indices/timestamps.
Output videos omit audio. Use a new output directory for each invocation.


In [ ]:
RUN_PREDICTION = False
INPUT = DRIVE / 'inference/example.jpg'
if RUN_PREDICTION:
    subprocess.run([sys.executable, '-m', 'aquafina_detector.predict', '--checkpoint', str(CHECKPOINT),
                    '--input', str(INPUT), '--output', str(DRIVE / 'predictions/example-v1'),
                    '--operating-point', str(operating_point)], check=True)
